<a href="https://colab.research.google.com/github/YChen1212/SystemAccount_Risk_Analysis/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# 匯入套件
import pandas as pd
import numpy as np
import random
from google.colab import files
from google.colab import drive

In [7]:
# 掛載 Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/帳號權限data/all調整.csv')

In [9]:
# 權限欄位清理，去掉不必要字元（\[]）
# df['permissions']=df['permissions'].replace(r'[\\\[\]]', '', regex=True)
df['permissions']=df['permissions'].replace(r'[\\\[\]]', '', regex=True)

In [10]:
# 權限欄位展開，每個權限獨立成一列
df['permissions'] = df['permissions'].str.split(',')
df = df.explode('permissions', ignore_index=True)

In [11]:
# 檢視各欄位之空值數量
df.isna().sum()

,0
user_id,0
user_name,0
user_account,0
department,0
job_title,0
permissions,0
is_active,0
status,0
last_logon_date,0
hire_date,0


In [12]:
# 檢視各欄位之資料型態
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   user_id          79 non-null     object
 1   user_name        79 non-null     object
 2   user_account     79 non-null     object
 3   department       79 non-null     object
 4   job_title        79 non-null     object
 5   permissions      79 non-null     object
 6   is_active        79 non-null     bool  
 7   status           79 non-null     object
 8   last_logon_date  79 non-null     object
 9   hire_date        79 non-null     object
 10  account_estd     79 non-null     object
 11  terminate_date   15 non-null     object
dtypes: bool(1), object(11)
memory usage: 7.0+ KB


In [13]:
# 檢視數值型欄位的統計摘要
df.describe()

,user_id,user_name,user_account,department,job_title,permissions,is_active,status,last_logon_date,hire_date,account_estd,terminate_date
count,79,79,79,79,79,79,79,79,79,79,79,15
unique,50,50,50,7,15,20,2,2,20,48,48,8
top,E0046,user0046,ERP0046,FIN,人資,'編製會計憑證',True,在職,2025-08-04,2022-09-30,2022-09-30,2025-08-02
freq,4,4,4,23,10,8,65,64,9,4,4,3


In [14]:
# 設定各部門與職位對應的權限，字典格式為{'部門':{'職稱':['權限']}}
dept_job = {
    'HR':{'人資主管':['薪資管理'], '人資':['招聘作業','員工資料維護']},
    'FIN':{'財務主管':['傳票核准'], '會計':['編製會計憑證','費用報支審核'], '出納':['付款','出納確認'] },
    'ITS':{'資訊主管':['SUPER_USER'], '資訊人員':['SUPER_USER']},
    'SALES':{'銷售主管':['價格審核','銷售報表查詢'], '銷售人員':['銷售訂單維護','客戶資料維護']},
    'PD':{'採購主管':['採購審核'],'採購人員':['採購訂單輸入','產品定價管理']},
    'R&D':{'研發主管':['研發文件審核'], '研發人員':['研發專案資料維護']},
    'IA':{'主任稽核':['查閱'], '稽核人員':['查閱']}
    }

In [15]:
# 新增欄位計算

# (1) 已離職但帳號仍啟用
df['offboard_but_active'] = (
    df['terminate_date'].notna() &
    (df['is_active'] == True)
)

# (2) 超過3個月未登入(假設查核當日為2025/08/09)
today = pd.to_datetime('2025-08-09')
df['no_login_3m'] = pd.to_datetime(df['last_logon_date']) < (today - pd.DateOffset(months=3))

# (3) 帳號建立日早於到職日
df['estd_earlythan_hire'] = pd.to_datetime(df['hire_date'])>pd.to_datetime(df['account_estd'])

# (4) 檢查各部門職稱是否符合應有的權限
def check_permission(row):
    dept = row['department'].strip()
    job = row['job_title'].strip()
    perm = row['permissions'].strip()


    if dept in dept_job:
        if job in dept_job[dept]:
            if perm in dept_job[dept][job]:
              return False # 權限符合，無異常
    return True # 權限不符合，異常

df['permission_valid'] = df.apply(check_permission, axis=1)

In [16]:
# 匯出清理過之資料為csv檔並下載到本地
df.to_csv('Final.csv', index=False)
files.download('Final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>